# Nettoyage de la base d'apprentissage

In [58]:
import numpy as np
import pandas as pd
import sys
import os
import dill as pickle

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
# On charge la base d'apprentissage et celle des classements FIFA
df = pd.read_csv("../data_finale/base_apprentissage.csv")
df_fifa = pd.read_csv("../data/classement_fifa/fifa_ranking_fin_saison.csv", sep=",", encoding="utf-8-sig")

In [60]:
df

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,...,12.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,...,23.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,...,4.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17186,ITA-Serie A,2526,Roma,Antonio Arena,ITA,FW,17-083,2009.0,2,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17187,ITA-Serie A,2526,Sassuolo,Luca Moro,ITA,FW,25-099,2001.0,14,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17188,ITA-Serie A,2526,Torino,Rafel Obrador,ESP,MF,22-069,2004.0,13,11,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
17189,ITA-Serie A,2526,Udinese,Idrissa Guèye,SEN,FW,19-230,2006.0,16,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Traitement des doublons

In [61]:
verifier_doublons_metier_et_techniques(df)

Recherche de doublons
 Attention : 787 lignes sont des doublons techniques stricts.
(Même joueur, même saison, même club -> Erreur d'extraction/jointure)

Exemple de lignes techniques concernées :
                player  season         team
28             Willian    2021      Arsenal
29             Willian    2021      Arsenal
30             Willian    2021      Arsenal
37   Emiliano Martínez    2021  Aston Villa
38   Emiliano Martínez    2021  Aston Villa
119           Jorginho    2021      Chelsea

872 lignes correspondent à des doublons de mercato
(Même joueur, même saison, mais clubs différents -> Transferts de mi-saison)

Exemple de joueurs transférés concernés :
                    player  season     team
0   Ainsley Maitland-Niles    2021  Arsenal
14             Joe Willock    2021  Arsenal
16         Martin Ødegaard    2021  Arsenal
17             Mathew Ryan    2021  Arsenal
25          Sead Kolašinac    2021  Arsenal
26        Shkodran Mustafi    2021  Arsenal


{'doublons_techniques': np.int64(787), 'doublons_mercato': np.int64(872)}

In [62]:
df = fusionner_doublons_techniques(df)

Format initial de la base : (17191, 123)
Format après fusion intelligente des doublons : (16404, 123)


In [63]:
df = fusionner_et_recalculer_mercato(df)

Format avant fusion mercato : (16404, 123)
Format après fusion mercato : (15532, 123)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  if colonnes_dates is None:
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  if colonnes_dates is None:


## Homogénéisation des formats

In [64]:
colonnes_dates = ["date_of_birth", "contract_expiration_date"]

# Application ciblée
df = nettoyer_age_et_dates(
    df, colonnes_dates=colonnes_dates
)

Traitement des colonnes de dates : ['date_of_birth', 'contract_expiration_date']
 -> Toutes les heures ont été remises à minuit.
 -> Colonne 'born' (année de naissance) extraite.
Calcul et nettoyage de la colonne 'age'...
 -> Attention : 0 âges manquants remplacés par la médiane (25 ans).
 -> Colonne 'age' convertie strictement en entiers (int).
Nettoyage de l'âge et des dates terminé.



## Traitement des variables avec beaucoup de valeurs manquantes

In [65]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 94.6% de valeurs manquantes
   • Performance_CS% : 92.8% de valeurs manquantes
   • Performance_Save% : 92.6% de valeurs manquantes
   • Performance_W : 92.4% de valeurs manquantes
   • Performance_D : 92.4% de valeurs manquantes
   • Performance_L : 92.4% de valeurs manquantes
   • Performance_Saves : 92.4% de valeurs manquantes
   • Performance_GA90 : 92.4% de valeurs manquantes
   • Performance_GA : 92.4% de valeurs manquantes
   • Performance_SoTA : 92.4% de valeurs manquantes
   • Penalty Kicks_PKm : 92.4% de valeurs manquantes
   • Penalty Kicks_PKsv : 92.4% de valeurs manquantes
   • Penalty Kicks_PKatt : 92.4% de valeurs manquantes
   • Performance_CS : 92.4% de valeurs manquantes
   • Penalty Kicks_PKA : 92.4% de valeurs manquantes
   • np_xg : 47.3% de valeurs manquantes
   • xa : 47.3% de valeurs

In [66]:
# Application de la fonction
df = nettoyer_valeurs_manquantes_ciblees(df)

Début du traitement ciblé des valeurs manquantes...
 -> 25 colonnes de performance nettoyées (NaN -> 0).
 -> Propagation inter-saisons terminée pour 6 colonnes fixes.
Finitions terminées (Derniers NaN résiduels convertis en valeurs neutres).


In [67]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • xa : 47.3% de valeurs manquantes
   • xg : 47.3% de valeurs manquantes
   • xg_buildup : 47.3% de valeurs manquantes
   • xg_chain : 47.3% de valeurs manquantes
   • np_xg : 47.3% de valeurs manquantes
   • date : 17.2% de valeurs manquantes
   • market_value_in_eur : 17.2% de valeurs manquantes
   • valuation_season_year : 17.2% de valeurs manquantes
   • contract_expiration_date : 15.7% de valeurs manquantes
   • Subs_Mn/Sub : 14.2% de valeurs manquantes
   • Starts_Mn/Start : 11.1% de valeurs manquantes
   • height_in_cm : 6.9% de valeurs manquantes
   • born : 6.9% de valeurs manquantes

Total : 13 colonnes dépassent le seuil de 1%.


## Encodage de variables

In [68]:
# Analyser la base
var_categorielles = lister_variables_categorielles(df)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • player (5553 modalités uniques)
   • team (137 modalités uniques)
   • league (5 modalités uniques)
   • nation (129 modalités uniques)
   • pos (10 modalités uniques)
   • join_key (5551 modalités uniques)
   • match_method (24 modalités uniques)
   • name (4647 modalités uniques)
   • sub_position (13 modalités uniques)
   • position (5 modalités uniques)
   • foot (3 modalités uniques)
   • date (256 modalités uniques)

Total : 12 variables catégorielles trouvées.


In [69]:
# Variables catégorielles à traiter
mes_variables = ["pos", "sub_position", "nation", "league", "foot"]

# Lancement de l'encodage
df = encoder_dataset_football(
    df=df,
    colonnes_categoriques=mes_variables,
    df_fifa_historique=df_fifa,
)

Format initial avant encodage : (15532, 123)
Encodage de 'nation' en 10 colonnes binaires Top FIFA (par saison)...
   • Variable 'confederation' encodée en colonnes binaires.
   • Les 10 colonnes classement_FIFA_X ont été injectées.
Profil Joueurs de champ détecté : Encodage Multi-Label de 'pos'.
Encodage One-Hot des colonnes : ['sub_position', 'league', 'foot']
Format final après encodage : (15532, 160)



## Jours contrat restants

In [70]:
df = calculer_jours_contrat_restants(df)

Début du calcul de la durée restante des contrats...
Contrats déjà expirés (valeur négative) : 17
Contrats manquants (NaN)                  : 2439

Statistiques descriptives de la variable calculée :
count    13093.000000
mean      1400.672420
std        740.074736
min      -1096.000000
25%        731.000000
50%       1461.000000
75%       1826.000000
max       5113.000000
Calcul terminé avec succès.


### PROBLÈME : LES DATES DE FIN DE CONTRAT NE SONT PAS LES BONNES

**Constat :** Les informations de fin de contrat actuellement présentes dans notre dataset correspondent aux données **actuelles** extraites rétroactivement (par exemple, un contrat courant jusqu'au *30 juin 2035*). 

**Impact :** Cela fausse le calcul de la variable `contrat_jours_restants`.

## Suppression de colonnes en double

In [71]:
colonnes_redondantes = [
    "Starts_Starts", "Standard_PK", "Standard_PKatt", "Standard_Gls", "90s", "Playing Time_Min%",
    "Performance_SoTA", "Performance_G+A", "Team Success_+/-", "Team Success_+/-90", "Playing Time_Min", 
    "Penalty Kicks_PKatt", "born", "np_xg", "xg_chain", "Per 90 Minutes_G+A-PK", "Per 90 Minutes_G-PK",
    "join_key", "tm_join_key", "tm_join_key_full", "tm_id", "player_id", "dob_key", "tm_dob_key",
    "match_method", "name", "date_of_birth", "dob_year", "date", "season", "valuation_season_year",
    "contract_expiration_date", "Starts_Mn/Start", "Performance_W",
    "Performance_D", "Performance_L"
]

df = supprimer_colonnes_du_dataset(df, colonnes_redondantes)

32 colonne(s) supprimée(s) : ['Starts_Starts', 'Standard_PK', 'Standard_PKatt', 'Standard_Gls', '90s', 'Playing Time_Min%', 'Performance_SoTA', 'Performance_G+A', 'Team Success_+/-', 'Team Success_+/-90', 'Playing Time_Min', 'Penalty Kicks_PKatt', 'born', 'np_xg', 'xg_chain', 'Per 90 Minutes_G+A-PK', 'Per 90 Minutes_G-PK', 'join_key', 'tm_id', 'player_id', 'dob_key', 'match_method', 'name', 'date_of_birth', 'date', 'season', 'valuation_season_year', 'contract_expiration_date', 'Starts_Mn/Start', 'Performance_W', 'Performance_D', 'Performance_L']


## Sauvegarde de la pipeline de premier nettoyage

In [72]:
# Sauvegarde de la fonction dans le fichier .pkl
with open("../data_finale/pipelines/pipeline_nettoyage.pkl", "wb") as fichier:
    pickle.dump(executer_pipeline_nettoyage, fichier)

## Traitement des outliers et normalisations

In [73]:
dossier_sortie = r"..\data_finale"

cols_a_normaliser = [
    "market_value_in_eur",
    "Playing Time_MP",
    "Playing Time_Starts",
    "Starts_Compl",
    "Subs_Subs",
    "Subs_Mn/Sub",
    "Subs_unSub",
    "Performance_Gls",
    "Performance_Ast",
    "Performance_PK",
    "Performance_PKatt",
    "Performance_Saves",
    "Performance_Save%",
    "Performance_CS",
    "Performance_CS%",
    "Standard_Sh",
    "Standard_SoT",
    "Standard_SoT%",
    "Standard_G/Sh",
    "Standard_G/SoT",
    "Team Success_PPM",
    "xg",
    "xa",
    "xg_buildup",
]

In [74]:
df_train, df_val, df_test = executer_pipeline_preprocessing(
    df = df,
    dossier_sortie=dossier_sortie,
    cols_a_normaliser=cols_a_normaliser,
    methode_split="temporel",
    height_min=155,
    height_max=210,
)

2107 lignes de la saison 2025 ont été écartées du pipeline.
Split effectué -> Train: 8125 | Val: 2633 | Test: 2667
Outliers traités (Bornes: [155, 210] | Remplacement par la médiane du poste du Train).
Pipeline terminé ! Fichiers sauvegardés dans : ..\data_finale



## Sauvegarde de la pipeline de second nettoyage

In [75]:
# Sauvegarde de la fonction dans le fichier .pkl
with open("../data_finale/pipelines/pipeline_anti_data_leakage.pkl", "wb") as fichier:
    pickle.dump(executer_pipeline_preprocessing, fichier)